<a href="https://colab.research.google.com/github/Chanthul4054/Telco-Customer-Churn-Prediction/blob/main/Course_Work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Customer Churn Prediciton

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score, roc_auc_score,confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models


##Load and Clean Data

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Telco Dataset/Telco-Customer-Churn-dataset.csv')
print(df.shape)
df.info()
df.describe()
display(df.describe(include="all").T)

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace = True)
if 'customerID' in df.columns:
  df.drop('customerID', axis = 1, inplace = True)
if df['Churn'].dtype == 'object':
  df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df.head()

##Exploratory Data Analysis (EDA)

Numeric Feature Distributions

In [ ]:

sns.histplot(df['tenure'], bins=30, kde=True , color = 'skyblue')
plt.title('Distribution of Tenure[Months]')
plt.xlabel('Tenure')
plt.show()

sns.histplot(df['MonthlyCharges'], bins= 30, kde= True, color = 'orange')
plt.title('Distribution of Monthly Charges')
plt.xlabel('Monthly Charges ($)')
plt.show()

sns.histplot(df['TotalCharges'], bins=30, kde=True, color = 'green')
plt.title('Distribution of Total Charges')
plt.xlabel('Total Charges ($)')
plt.show()

Numerical Features vs. Churn (Box Plots)

In [ ]:
sns.boxplot(x='Churn', y = 'tenure', data = df)
plt.title('Tenure vs. Churn')
plt.xlabel(['No Churn' , 'Churn'])
plt.show()

sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges vs Churn')
plt.xlabel(['No Churn', 'Churn'])
plt.show()

sns.boxplot(x='Churn', y='TotalCharges', data=df)
plt.title('Total Charges vs Churn')
plt.xlabel(['No Churn', 'Churn'])
plt.show()

Categorical Features vs. Churn

In [ ]:
sns.countplot(x='PaymentMethod' , hue = 'Churn', data = df)
plt.title('Churn by Payment Method')
plt.tick_params(axis='x', rotation=45)
plt.show()

sns.countplot(x='InternetService', hue='Churn', data=df)
plt.title('Churn by Internet Service Type')
plt.show()

sns.countplot(x='SeniorCitizen', hue='Churn', data=df)
plt.title('Churn by Senior Citizen Status')
plt.xlabel('Senior Citizen (1 = Yes, 0 = No)')
plt.show()


#Data Preprocessing

In [ ]:
#Separate Features
x = df.drop('Churn', axis = 1)
y = df['Churn']
x = pd.get_dummies(x, drop_first = True)

# Train-test split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2 , random_state=42 , stratify=y )

#scaling
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)



#Decision Tree Model

In [ ]:
dt = DecisionTreeClassifier(random_state=42)

param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='accuracy')
grid_search.fit(x_train_scaled, y_train)

best_dt = grid_search.best_estimator_

y_pred = best_dt.predict(x_test_scaled)

print(f"Best DT Params: {grid_search.best_params_}")
print("\n--- Decision Tree Classification Report ---")
print(classification_report(y_test, y_pred))

dt_roc_auc = roc_auc_score(y_test, best_dt.predict_proba(x_test_scaled)[:, 1])
print(f"Decision Tree ROC AUC Score: {dt_roc_auc}")

